# Context-1 Colab Server

This notebook serves `chromadb/context-1` from Colab, exposes an agent-style inference API, and is designed to sit behind Cloudflare so the local harness can call `https://context1.pkking.computer`.

## Expected secrets

- `HF_TOKEN`
- `CLOUDFLARE_TUNNEL_TOKEN`
- Optional: `MODEL_NAME` (defaults to `chromadb/context-1`)
- Optional: `PUBLIC_HOSTNAME` (defaults to `context1.pkking.computer`)
- Optional: `VLLM_PORT` (defaults to `8001`)
- Optional: `API_PORT` (defaults to `8000`)

In [ ]:
import os
from pathlib import Path

MODEL_NAME = os.getenv('MODEL_NAME', 'chromadb/context-1')
PUBLIC_HOSTNAME = os.getenv('PUBLIC_HOSTNAME', 'context1.pkking.computer')
VLLM_PORT = int(os.getenv('VLLM_PORT', '8001'))
API_PORT = int(os.getenv('API_PORT', '8000'))
WORKDIR = Path('/content/context1-server')
WORKDIR.mkdir(parents=True, exist_ok=True)
print({'MODEL_NAME': MODEL_NAME, 'PUBLIC_HOSTNAME': PUBLIC_HOSTNAME, 'VLLM_PORT': VLLM_PORT, 'API_PORT': API_PORT})

In [ ]:
!apt-get update -y
!apt-get install -y cloudflared
!pip install -q --upgrade pip
!pip install -q vllm fastapi uvicorn requests sse-starlette huggingface_hub

In [ ]:
from huggingface_hub import login

hf_token = os.environ['HF_TOKEN']
login(token=hf_token)

In [ ]:
server_code = r'''
import json
import os
from typing import Any, Dict, List

import requests
from fastapi import FastAPI
from fastapi.responses import JSONResponse, StreamingResponse

VLLM_BASE_URL = f"http://127.0.0.1:{os.getenv('VLLM_PORT', '8001')}/v1"
MODEL_NAME = os.getenv('MODEL_NAME', 'chromadb/context-1')

app = FastAPI(title='Context-1 Agent Server')

def trajectory_to_messages(trajectory: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    messages = []
    for event in trajectory:
        role = event.get('role', 'user')
        content = event.get('content', '')
        if isinstance(content, (dict, list)):
            content = json.dumps(content)
        messages.append({'role': role, 'content': content})
    return messages or [{'role': 'user', 'content': 'Search for relevant supporting documents.'}]

def offline_payload() -> Dict[str, Any]:
    return {
        'error': {
            'type': 'service_unavailable',
            'message': 'The Colab-backed Context-1 runtime is not ready yet.'
        }
    }

def stream_vllm(payload: Dict[str, Any]):
    response = requests.post(f"{VLLM_BASE_URL}/chat/completions", json=payload, stream=True, timeout=600)
    response.raise_for_status()
    for line in response.iter_lines(decode_unicode=True):
        if not line:
            continue
        if line.startswith('data: '):
            yield line + '\n\n'
    yield 'data: [DONE]\n\n'

@app.get('/healthz')
def healthz():
    try:
        response = requests.get(f"{VLLM_BASE_URL}/models", timeout=15)
        response.raise_for_status()
        return {'status': 'ok', 'model': MODEL_NAME, 'provider': 'vllm'}
    except Exception as exc:
        return JSONResponse(status_code=503, content={'status': 'offline', 'error': str(exc)})

@app.post('/v1/agent/step')
def agent_step(payload: Dict[str, Any]):
    trajectory = payload.get('trajectory', [])
    tools = payload.get('tools', [])
    generation = payload.get('generation', {})
    request_payload = {
        'model': generation.get('model', MODEL_NAME),
        'messages': trajectory_to_messages(trajectory),
        'temperature': generation.get('temperature', 0),
        'max_tokens': generation.get('max_tokens', 1024),
        'stream': payload.get('stream', True),
    }
    if tools:
        request_payload['tools'] = tools
    if payload.get('stream', True):
        return StreamingResponse(stream_vllm(request_payload), media_type='text/event-stream')
    try:
        response = requests.post(f"{VLLM_BASE_URL}/chat/completions", json=request_payload, timeout=600)
        response.raise_for_status()
        return response.json()
    except Exception:
        return JSONResponse(status_code=503, content=offline_payload())
'''

server_path = WORKDIR / 'context1_agent_server.py'
server_path.write_text(server_code)

In [ ]:
import subprocess

vllm_cmd = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL_NAME,
    '--host', '127.0.0.1',
    '--port', str(VLLM_PORT),
]
vllm_proc = subprocess.Popen(vllm_cmd, cwd=WORKDIR)
api_proc = subprocess.Popen([
    'python', '-m', 'uvicorn', 'context1_agent_server:app',
    '--host', '0.0.0.0', '--port', str(API_PORT)
], cwd=WORKDIR, env={**os.environ, 'MODEL_NAME': MODEL_NAME, 'VLLM_PORT': str(VLLM_PORT)})
print({'vllm_pid': vllm_proc.pid, 'api_pid': api_proc.pid})

In [ ]:
import time
import requests

for _ in range(60):
    try:
        response = requests.get(f'http://127.0.0.1:{API_PORT}/healthz', timeout=10)
        print(response.json())
        if response.ok:
            break
    except Exception as exc:
        print('waiting:', exc)
    time.sleep(10)

In [ ]:
import subprocess

cloudflared_proc = subprocess.Popen([
    'cloudflared', 'tunnel', 'run', '--token', os.environ['CLOUDFLARE_TUNNEL_TOKEN']
])
print({'cloudflared_pid': cloudflared_proc.pid, 'public_hostname': PUBLIC_HOSTNAME})

In [ ]:
sample_payload = {
    'trajectory': [
        {'role': 'system', 'content': 'You are Context-1, a search subagent.'},
        {'role': 'user', 'content': 'Find documents about Chroma Context-1.'}
    ],
    'tools': [],
    'generation': {'model': MODEL_NAME, 'temperature': 0, 'max_tokens': 256},
    'stream': False,
}

import requests
response = requests.post(f'http://127.0.0.1:{API_PORT}/v1/agent/step', json=sample_payload, timeout=600)
print(response.status_code)
print(response.text[:2000])